# Kiểm thử retrieval EXIT với 10.000 passage thật
Chạy tuần tự từng cell trên Kaggle, bật Internet; GPU T4 dùng cho bước encode câu hỏi. Notebook dùng thư mục riêng và không sửa dữ liệu full.

Luồng kiểm tra: passage/vector khớp ID → SQLite resume → sampling resume → PQ96 → index resume → verify → Contriever thật → so sánh PQ/Flat → export ctxs.

Cần có sẵn một shard embedding chính thức hoặc archive `wikipedia_embeddings.tar` đã tải xong, cùng corpus TSV/TSV.GZ hoặc SQLite đầy đủ. Không dùng file `.part`. Dù chỉ chọn 10.000 vector, pickle vẫn phải nạp cả một shard vào RAM.

Upload/restore Kaggle thật và EXIT/reader thật có cờ bật riêng. Phần bị tắt được ghi **SKIP**, không tính là đã kiểm chứng.


In [ ]:
from pathlib import Path
import os, sys, json, time, uuid, subprocess
REPO = Path('/kaggle/working/EXIT')  # sửa nếu checkout ở chỗ khác
assert (REPO / 'retrieval/smoke.py').exists(), 'Cập nhật cả thư mục retrieval, gồm smoke.py và progress.py'
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))


In [ ]:
%pip install -r requirements-retrieval.txt


## 1. Cấu hình riêng cho test
Sửa đường dẫn nguồn nếu tự dò không đúng. Mỗi lần chạy tạo thư mục mới; không xóa hoặc ghi đè lượt cũ.

`RUN_KAGGLE=True` tạo/cập nhật Dataset **public dành riêng cho test** và kiểm tra API trước tác vụ nặng. `RUN_EXIT=True` cần dependency, quyền truy cập model và Gemini Secret. Giữ `N=10_000` cho cấu hình notebook mặc định.


In [ ]:
RUN_KAGGLE = False  # bật để kiểm thử upload + restore thật
RUN_EXIT = False    # bật để chạy EXIT + Gemini thật sau retrieval
N = 10_000
RUN_ID = uuid.uuid4().hex[:8]
BASE = Path('/tmp/exit-retrieval-smoke') / RUN_ID
FIXTURE = BASE / 'fixture'
STATE = BASE / 'state'
STATE.mkdir(parents=True)
REPORT = Path('/kaggle/working') / f'retrieval-smoke-report-{RUN_ID}.json'
STARTED = time.monotonic()
checks = {'kaggle': 'SKIP', 'exit_reader': 'SKIP'}

# Để None để tự dò nguồn trong phiên hiện tại, hoặc gán đường dẫn cụ thể.
SOURCE_SHARD = None  # '/tmp/retrieval-sources/wikipedia_embeddings/wiki_00'
EMBEDDING_ARCHIVE = Path('/tmp/retrieval-downloads/wikipedia_embeddings.tar')
CORPUS_SOURCE = None # '/tmp/retrieval-downloads/psgs_w100.tsv.gz'
OWNER = json.loads(Path('/kaggle/working/retrieval-session.json').read_text()).get('owner', 'YOUR_USERNAME') if Path('/kaggle/working/retrieval-session.json').exists() else 'YOUR_USERNAME'
DATASET_PREFIX = 'exit-retrieval-smoke'  # khác Dataset full
config_data = {
    'owner': OWNER, 'dataset_prefix': DATASET_PREFIX, 'public': True,
    'root': str(STATE), 'downloads': str(BASE / 'unused-downloads'),
    'sources': str(FIXTURE), 'tsv': str(FIXTURE / 'small.tsv'),
    'shards': str(FIXTURE / 'parts/shard-*'),
    'dimension': 768, 'pq_m': 96, 'bits': 8, 'sample_size': N,
    'seed': 42, 'batch_size': 1000, 'threads': 4,
    'checkpoint_minutes': 120, 'time_budget_minutes': 600,
    'api_check_timeout_minutes': 10, 'upload_timeout_minutes': 30,
}
CONFIG_PATH = BASE / 'smoke-config.json'
CONFIG_PATH.write_text(json.dumps(config_data, indent=2))
print('Scratch:', BASE)
print('Report:', REPORT)
print('Kaggle public test:', RUN_KAGGLE, '| EXIT/reader:', RUN_EXIT)


In [ ]:
# Có thể chạy local tests mà không bật phần Kaggle.
if RUN_KAGGLE:
    assert OWNER != 'YOUR_USERNAME', 'Điền OWNER trước'
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['KAGGLE_API_TOKEN'] = secrets.get_secret('KAGGLE_API_TOKEN')
    subprocess.run([sys.executable, '-u', '-m', 'retrieval.prepare', 'check-api', '--config', str(CONFIG_PATH)], check=True)
    checks['api_check'] = 'PASS'
else:
    checks['api_check'] = 'SKIP'
    print('SKIP: Kaggle API/upload chưa được kiểm tra trong lượt này')


## 2. Chọn nguồn và tạo bộ nhỏ đúng ID
Notebook không tải toàn bộ dữ liệu. Nếu chưa có nguồn, chuẩn bị corpus và archive/shard trước rồi chỉnh đường dẫn. Corpus SQLite phải chứa đủ ID được chọn; dùng TSV.GZ nếu SQLite đang import dở. Có thể tách riêng một shard từ archive hoàn chỉnh mà không giải nén toàn bộ.


In [ ]:
import glob
from retrieval.smoke import extract_first_shard, subset

if SOURCE_SHARD is None:
    candidates = []
    for pattern in ('/tmp/retrieval-sources/**/passages_*', '/tmp/retrieval-sources/**/wiki_*'):
        candidates.extend(p for p in glob.glob(pattern, recursive=True)
                          if Path(p).is_file() and not p.endswith(('.tmp', '.faiss', '.json')))
    if candidates:
        SOURCE_SHARD = Path(sorted(candidates)[0])
    elif EMBEDDING_ARCHIVE.exists():
        SOURCE_SHARD = extract_first_shard(EMBEDDING_ARCHIVE, BASE / 'one-source-shard')
    else:
        raise FileNotFoundError('Chưa tìm thấy shard hoặc archive hoàn chỉnh. Gán SOURCE_SHARD hoặc EMBEDDING_ARCHIVE; không dùng .part.')
SOURCE_SHARD = Path(SOURCE_SHARD)
if CORPUS_SOURCE is None:
    candidates = [Path('/tmp/retrieval-sources/psgs_w100.tsv'),
                  Path('/tmp/retrieval-downloads/psgs_w100.tsv.gz')]
    CORPUS_SOURCE = next((p for p in candidates if p.exists()), None)
    if CORPUS_SOURCE is None:
        raise FileNotFoundError('Gán CORPUS_SOURCE tới TSV, TSV.GZ hoặc SQLite chứa đủ passage')
CORPUS_SOURCE = Path(CORPUS_SOURCE)
print('Embedding shard:', SOURCE_SHARD, f'({SOURCE_SHARD.stat().st_size / 1024**3:.2f} GiB)')
print('Corpus:', CORPUS_SOURCE)
ids, vectors = subset(SOURCE_SHARD, CORPUS_SOURCE, FIXTURE, count=N)
assert vectors.shape == (N, 768) and len(set(ids)) == N
# Hai shard nhỏ giúp kiểm tra sampling/index qua ranh giới shard.
import pickle
parts = FIXTURE / 'parts'
parts.mkdir()
for i, (start, end) in enumerate(((0, N // 2), (N // 2, N))):
    with (parts / f'shard-{i:02d}').open('wb') as f:
        pickle.dump((ids[start:end], vectors[start:end]), f, protocol=pickle.HIGHEST_PROTOCOL)
checks['subset_ids'] = 'PASS'


## 3. SQLite: dừng sau batch rồi tiếp tục
Không bấm Interrupt thật. Ta chủ động dừng sau một transaction, gọi lại cùng hàm rồi so toàn bộ ID/title/text với lượt import liên tục.


In [ ]:
import numpy as np
from retrieval.storage import prepare_corpus, Corpus, write_json, digest

partial_db = BASE / 'sqlite-resume.sqlite'
partial = prepare_corpus(FIXTURE / 'small.tsv', partial_db, deadline=lambda: True, batch_size=1000)
assert partial['count'] == 1000 and not partial['complete']
completed = prepare_corpus(FIXTURE / 'small.tsv', partial_db, batch_size=1000)
assert completed['count'] == N and completed['complete']
prepare_corpus(FIXTURE / 'small.tsv', STATE / 'corpus.sqlite')
a, b = Corpus(STATE / 'corpus.sqlite'), Corpus(partial_db)
try:
    assert a.get_many(ids) == b.get_many(ids)
    assert a.connection.execute('SELECT COUNT(*) FROM passages').fetchone()[0] == N
finally:
    a.close(); b.close()
checks['sqlite_resume'] = 'PASS'
print('PASS: SQLite khớp từng ID/title/text sau resume')


## 4. Sampling: so lượt liên tục với lượt có dừng
Dùng reservoir 2.000 vector để kiểm tra cả trạng thái RNG. Mẫu dùng train PQ ở bước sau vẫn là 10.000 vector.


In [ ]:
from types import SimpleNamespace
from retrieval import prepare

def make_args(work, output, sample_size=N):
    work.mkdir(parents=True, exist_ok=True)
    return SimpleNamespace(work=work, output=output, corpus=STATE / 'corpus.sqlite',
        shards=str(parts / 'shard-*'), dimension=768, pq_m=96, bits=8,
        sample_size=sample_size, seed=42, batch_size=1000, checkpoint_minutes=120, control=None)

sa = make_args(BASE / 'sampler-a', BASE / 'unused-a', sample_size=2000)
sb = make_args(BASE / 'sampler-b', BASE / 'unused-b', sample_size=2000)
prepare.sample(sa, lambda: False)
prepare.sample(sb, lambda: True)
prepare.sample(sb, lambda: False)
with (sa.work / 'sample.pkl').open('rb') as f: sample_a = pickle.load(f)
with (sb.work / 'sample.pkl').open('rb') as f: sample_b = pickle.load(f)
np.testing.assert_array_equal(sample_a['sample'], sample_b['sample'])
assert sample_a['seen'] == sample_b['seen'] == N
assert sample_a['rng'] == sample_b['rng']
checks['sampling_resume'] = 'PASS'
print('PASS: reservoir và RNG khớp sau resume')


## 5. Train PQ96 và dựng artifact A liên tục
Bước này chạy CPU, có thể mất vài phút. WORKING là heartbeat thời gian, không phải phần trăm hoàn thành. Không train lại Contriever.


In [ ]:
import faiss
faiss.omp_set_num_threads(4)
args_a = make_args(STATE / 'work', STATE / 'artifact')
prepare.sample(args_a, lambda: False)
prepare.train(args_a)
prepare.build(args_a, lambda: False)
prepare.verify(args_a.output)
manifest_a = json.loads((args_a.output / 'manifest.json').read_text())
assert manifest_a['vectors'] == manifest_a['passages'] == N
checks['artifact_verify'] = 'PASS'


## 6. Artifact B: dừng ở 4.000 vector rồi load checkpoint
Hai lượt dùng cùng bộ PQ đã train, cùng batch size và thứ tự ID. Đây là dừng có kiểm soát để kiểm tra resume; không chứng minh bấm Interrupt luôn lưu checkpoint kịp.


In [ ]:
import shutil
from utils.src.index import Indexer

b_root = BASE / 'index-resume'
args_b = make_args(b_root / 'work', b_root / 'artifact')
shutil.copytree(args_a.work / 'trained', args_b.work / 'trained')
for name in ('sample.pkl', 'sample.json'):
    shutil.copyfile(args_a.work / name, args_b.work / name)
class ExpectedStop(Exception):
    pass
class StopAt4000:
    def tick(self, save, progress):
        if progress['vectors'] >= 4000:
            save()
            raise ExpectedStop()
args_b.control = StopAt4000()
try:
    prepare.build(args_b, lambda: False)
except ExpectedStop:
    print('Đã dừng có chủ đích sau khi lưu checkpoint 4.000 vector')
else:
    raise AssertionError('Không xảy ra điểm dừng mong đợi')
generation = prepare.latest_checkpoint(args_b.work)
state = json.loads((generation / 'complete.json').read_text())
assert state['vectors'] == 4000
partial_index = Indexer(768)
partial_index.deserialize_from(str(generation))
assert partial_index.index.ntotal == 4000
assert partial_index.index_id_to_db_id == ids[:4000]
del partial_index
args_b.control = None
prepare.build(args_b, lambda: False)
prepare.verify(args_b.output)
qa, qb = Indexer(768), Indexer(768)
qa.deserialize_from(str(args_a.output)); qb.deserialize_from(str(args_b.output))
assert qa.index.ntotal == qb.index.ntotal == N
assert qa.index_id_to_db_id == qb.index_id_to_db_id == ids
for (ia, scores_a), (ib, scores_b) in zip(qa.search_knn(vectors[:20], 20), qb.search_knn(vectors[:20], 20)):
    assert ia == ib
    np.testing.assert_array_equal(scores_a, scores_b)
checks['index_resume'] = 'PASS'
print('PASS: index A/B cùng số vector, thứ tự ID và kết quả tìm kiếm')


## 7. Contriever thật → PQ/Flat → SQLite
Câu hỏi tạo từ title chỉ để kiểm tra luồng và xem kết quả, không phải benchmark QA có nhãn. Không yêu cầu PQ giống Flat hoàn toàn.

Overlap@K đo ảnh hưởng nén trên tập nhỏ, không thay cho answer-hit@K. Kiểm tra lưu/load dùng cùng vector câu hỏi để tách sai số encode khỏi việc lưu index.


In [ ]:
from retrieval.runtime import ContrieverRetriever

retriever = ContrieverRetriever(args_a.output)
rows = retriever.corpus.get_many(ids)
titles = list(dict.fromkeys(row[1] for row in rows if row[1]))[:20]
assert titles, 'Corpus không có title để tạo câu hỏi smoke'
queries = [f'What is {title}?' for title in titles]
query_vectors = retriever.encoder.encode(queries)
assert query_vectors.shape == (len(queries), 768)
assert np.isfinite(query_vectors).all()
flat = Indexer(768)
flat.index_data(ids, vectors)
flat_results = flat.search_knn(query_vectors, 20)
pq_results = retriever.index.search_knn(query_vectors, 20)
hits = retriever.retrieve_batch(queries, top_k=20)
assert len(hits) == len(queries)
for batch, (expected_ids, scores) in zip(hits, pq_results):
    assert len(batch) == 20 and [h.doc_id for h in batch] == expected_ids
    assert np.all(np.diff(scores) <= 1e-5)
    source_rows = retriever.corpus.get_many(expected_ids)
    assert [(h.doc_id, h.title, h.text) for h in batch] == source_rows
pq_overlap = {str(k): float(np.mean([len(set(p[0][:k]) & set(f[0][:k])) / k
                    for p, f in zip(pq_results, flat_results)])) for k in (5, 20)}
print('PQ/Flat overlap@K:', pq_overlap)
for q, batch in list(zip(queries, hits))[:5]:
    print('\nQUESTION:', q)
    print('TOP-1:', batch[0].doc_id, batch[0].title, batch[0].score)
    print(batch[0].text[:300])
# Lưu/load lại; dùng cùng vector câu hỏi để kiểm tra index.
reloaded = ContrieverRetriever(args_a.output, encoder=retriever.encoder)
for before, after in zip(pq_results, reloaded.index.search_knn(query_vectors, 20)):
    assert before[0] == after[0]
    np.testing.assert_array_equal(before[1], after[1])
reloaded.close()
checks['real_query_encoder'] = 'PASS'
checks['lookup_order_and_roundtrip'] = 'PASS'


## 8. Export ctxs có resume
Input không có đáp án vàng: `answers=[]` chỉ kiểm tra định dạng, không dùng để tuyên bố điểm QA. Lượt B dừng sau một batch rồi tiếp tục; so với lượt A.


In [ ]:
from retrieval.run import export_records
question_file = BASE / 'questions.jsonl'
question_file.write_text(''.join(json.dumps({'question': q, 'answers': []}) + '\n' for q in queries))
export_a, export_b = BASE / 'retrieved-a.jsonl', BASE / 'retrieved-b.jsonl'
assert export_records(retriever, question_file, export_a, batch_size=4)
assert not export_records(retriever, question_file, export_b, batch_size=4, deadline=lambda: True)
assert export_records(retriever, question_file, export_b, batch_size=4)
assert export_a.read_bytes() == export_b.read_bytes()
records = [json.loads(line) for line in export_a.read_text().splitlines()]
assert len(records) == len(queries)
assert all(len(r['ctxs']) == 20 and {'id','title','text','score'} <= set(r['ctxs'][0]) for r in records)
checks['ctxs_export_resume'] = 'PASS'
print('PASS: JSONL ctxs của hai lượt giống nhau')


## 9. Upload/restore Dataset thật — nếu bật RUN_KAGGLE
Dùng uploader hiện tại với Dataset test riêng cho corpus, checkpoint và artifact. Restore sang thư mục mới, verify hash và tìm kiếm lại.

Đây là kiểm tra trong cùng phiên. Vẫn cần chạy cell cuối ở một phiên Kaggle mới để xác nhận lưu trữ bền vững.


In [ ]:
if RUN_KAGGLE:
    from retrieval.workflow import Workflow, load_config, restore
    config = load_config(CONFIG_PATH)
    flow = Workflow(config)
    flow.check_api()
    flow.registry['completed'] = ['prepare-corpus','sample-pq','train-pq','build-index','verify']
    flow.save_registry()
    flow.publish_checkpoint()
    flow.publish_artifact()
    flow.registry['completed'].append('publish-artifact')
    flow.save_registry()
    flow.publish_checkpoint()  # lưu tham chiếu artifact version vào checkpoint
    restored_config = dict(config, root=str(BASE / 'remote-restored'))
    restore(restored_config)
    restored_artifact = Path(restored_config['root']) / 'artifact'
    prepare.verify(restored_artifact)
    for name, expected in manifest_a['files'].items():
        assert digest(restored_artifact / name) == expected
    restored_retriever = ContrieverRetriever(restored_artifact, encoder=retriever.encoder)
    remote_ids, remote_scores = restored_retriever.index.search_knn(query_vectors[:1], 20)[0]
    assert remote_ids == pq_results[0][0]
    np.testing.assert_array_equal(remote_scores, pq_results[0][1])
    assert restored_retriever.corpus.get_many(remote_ids) == retriever.corpus.get_many(remote_ids)
    restored_retriever.close()
    checks['kaggle'] = 'PASS (same-session roundtrip; new-session check still required)'
    print('PASS: upload + restore thật. Ghi lại owner/prefix:', OWNER, DATASET_PREFIX)
else:
    print('SKIP: chưa kiểm thử Kaggle upload/restore')


## 10. EXIT + reader thật — nếu bật RUN_EXIT
Bước này tải model EXIT/Gemma; cần quyền Hugging Face nếu model yêu cầu và `GEMINI_API_KEY` trong Kaggle Secrets. Package bổ sung chỉ cài khi bật.

Dùng Top-5 và một câu hỏi để kiểm tra giao diện, không chấm accuracy khi chưa có đáp án vàng. Nếu model gated cần token, đặt `HF_TOKEN` từ Secrets trước khi chạy.


In [ ]:
if RUN_EXIT:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'peft', 'accelerate', 'bitsandbytes', 'spacy', 'sentencepiece'], check=True)
    from kaggle_secrets import UserSecretsClient
    os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')
    # Nếu cần model gated: đặt HF_TOKEN từ Secret trước khi load.
    from exit_rag import ExitRAG, Document
    from gemini_reader import GeminiQuota
    import torch
    reader_config = json.loads(Path('configs/gemini_api.json').read_text())
    rag = ExitRAG(device='cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0',
        reader_backend='gemini', gemini_model=reader_config['model'],
        gemini_project_id=reader_config['project_id'], gemini_quota=GeminiQuota(**reader_config['quotas']),
        gemini_quota_state=BASE / 'gemini-quota.json', cache_dir=str(BASE / 'model-cache'))
    docs = [Document(h.title, h.text, h.score) for h in hits[0][:5]]
    result = rag.run_rag(queries[0], docs)
    assert isinstance(result['answer'], str) and result['answer'].strip()
    assert isinstance(result['compressed_context'], str)
    print('QUESTION:', queries[0])
    print('ANSWER:', result['answer'])
    write_json(BASE / 'exit-answer.json', result)
    checks['exit_reader'] = 'PASS (execution; answer correctness not scored)'
    # Runner thật đọc JSONL vừa export, không chạy lại cả benchmark.
    from reproduce import get_documents
    assert len(get_documents(records[0], top_k=5)) == 5
    checks['evaluation_reader_schema'] = 'PASS'
else:
    checks['evaluation_reader_schema'] = 'SKIP (ctxs schema assertions passed separately)'
    print('SKIP: chưa chạy EXIT/reader thật')


## 11. Báo cáo PASS/SKIP
Chỉ ghi PASS cho phần thực sự đã chạy. Báo cáo nhỏ được lưu trong Output. `local_checks_complete` cho biết các kiểm tra local bắt buộc đã chạy đủ chưa.

Kết quả không bảo đảm full Wikipedia vừa RAM hoặc thời gian. Bước tiếp theo vẫn là benchmark một shard thật và đánh giá answer-hit@K. Nếu cell lỗi, xử lý lỗi trước khi coi lượt kiểm thử hoàn chỉnh.


In [ ]:
import resource
required = ('subset_ids', 'sqlite_resume', 'sampling_resume', 'artifact_verify', 'index_resume', 'real_query_encoder', 'lookup_order_and_roundtrip', 'ctxs_export_resume')
report = {'local_checks_complete': all(checks.get(k) == 'PASS' for k in required), 'run_id': RUN_ID, 'passages': N, 'scratch': str(BASE), 'checks': checks,
          'pq_flat_overlap': pq_overlap, 'elapsed_seconds': time.monotonic() - STARTED,
          'process_peak_rss_gib': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024**2,
          'artifact_sizes': {p.name: p.stat().st_size for p in args_a.output.iterdir() if p.is_file()},
          'dataset_prefix': DATASET_PREFIX if RUN_KAGGLE else None}
write_json(REPORT, report)
print(json.dumps(report, ensure_ascii=False, indent=2))
print('Saved report:', REPORT)
retriever.close()


## 12. Kiểm tra qua phiên Kaggle mới — chạy cell dưới độc lập
Sau khi phần remote đã PASS, lưu báo cáo và kết thúc phiên. Trong phiên mới có repo/dependencies, sửa username/prefix trong cell dưới và chạy.

Cell không phụ thuộc biến RAM từ các cell trước, không tải embedding và không dựng lại index. Token lấy từ Secrets; checkpoint chỉ tới đúng version artifact. `sample_size=10000` phải khớp N của phiên tạo artifact.


In [ ]:
# Đổi thành True chỉ khi đã sang phiên mới và đã upload ở bước 9.
VERIFY_NEW_SESSION = False
if VERIFY_NEW_SESSION:
    import os, sys, json, uuid
    from pathlib import Path
    repo = Path('/kaggle/working/EXIT')
    os.chdir(repo)
    if str(repo) not in sys.path: sys.path.insert(0, str(repo))
    from kaggle_secrets import UserSecretsClient
    os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
    from retrieval.storage import write_json
    from retrieval.workflow import load_config, restore
    from retrieval.prepare import verify
    from retrieval.runtime import ContrieverRetriever
    settings = json.loads(Path('configs/retrieval_kaggle.json').read_text())
    settings.update(owner='YOUR_USERNAME', dataset_prefix='exit-retrieval-smoke', sample_size=10000,
                    root='/tmp/exit-smoke-new-session-' + uuid.uuid4().hex[:8])
    assert settings['owner'] != 'YOUR_USERNAME', 'Điền username đã dùng ở phiên trước'
    path = Path('/kaggle/working/smoke-restore-config.json')
    write_json(path, settings)
    cfg = load_config(path)
    restore(cfg)
    artifact = Path(cfg['root']) / 'artifact'
    verify(artifact)
    rt = ContrieverRetriever(artifact)
    sample_rows = rt.corpus.connection.execute('SELECT doc_id,title,text FROM passages ORDER BY doc_id LIMIT 1').fetchall()
    question = f'What is {sample_rows[0][1]}?'
    restored_hits = rt.retrieve(question, 5)
    assert len(restored_hits) == 5
    print('PASS: phiên mới đọc artifact, encode câu hỏi và retrieval được')
    print([(h.doc_id, h.title) for h in restored_hits])
    write_json(Path('/kaggle/working/smoke-new-session-report.json'), {'status':'PASS','question':question,'artifact':str(artifact)})
    rt.close()
else:
    print('SKIP: new-session verification chưa chạy')
